# Special Event Operations — Pedder scheduled audit processor

This ArcGIS Notebook scheduled task uses `GIS("home")`, the existing source item `53d7f0d643a14d1a8de20ca6e687f21a` layer 0, and the existing **single-layer** audit item `9fb69f32b67242e8a5ba093b0bfab9ae` layer 0.

It reconstructs the latest state per `SourceObjectID` from the ordered append-only audit log, appends `ADD`, `UPDATE`, or `DELETE` rows, retains last-known geometry on deletes, and uses a deterministic `DeliveryKey` for retry safety. Movement distance is calculated between the previous audit point and current source point; deletes and moves of at least 100 m can optionally create Workflow Manager jobs in item `8c3a1add5d444b089e9d077e3e940962`.

Keep `DRY_RUN = True` until permissions, service field types/lengths, spatial reference, source counts, and WFM behavior have been tested in a non-production environment. No token or password is stored here.

In [ ]:
from arcgis.gis import GIS
from datetime import datetime, timezone, timedelta
from contextlib import contextmanager
import hashlib
import json
import math
import os
import socket
import time
import uuid
from urllib.parse import quote
import requests

PORTAL_URL = "https://publicsafety.maps.arcgis.com"
SOURCE_ITEM_ID = "53d7f0d643a14d1a8de20ca6e687f21a"
SOURCE_LAYER_ID = 0
AUDIT_ITEM_ID = "9fb69f32b67242e8a5ba093b0bfab9ae"
AUDIT_LAYER_ID = 0
AUDIT_LAYER_URL = "https://services6.arcgis.com/SdBwj74JF24xKhpX/arcgis/rest/services/Special_Event_Asset_Audit_Log_20261030_144101/FeatureServer/0"
WORKFLOW_ITEM_ID = "8c3a1add5d444b089e9d077e3e940962"

SOURCE_FIELDS = {
    "unit_id": "unitid",
    "event_id": "eventid",
    "event_asset_id": "uniqueid",
    "unit_location": "unitloc",
    "unit_type": "unittype",
    "post_status": "poststatus",
    "staff_agency": "staffagency",
}
AUDIT_FIELDS = {
    "oid": "OBJECTID",
    "global_id": "GlobalID",
    "source_oid": "SourceObjectID",
    "source_global_id": "SourceGlobalID",
    "event_id": "EventID",
    "unit_id": "UnitID",
    "event_asset_id": "EventAssetID",
    "unit_location": "UnitLocation",
    "unit_type": "UnitType",
    "change_type": "ChangeType",
    "change_time": "ChangeDateTime",
    "editor_user": "EditorUser",
    "processed_time": "ProcessedDateTime",
    "delivery_key": "DeliveryKey",
    "source_url": "SourceLayerURL",
    "is_last_known": "IsLastKnown",
    "wfm_job_id": "WFMJobID",
    "wfm_status": "WFMStatus",
    "wfm_attempt_count": "WFMAttemptCount",
    "wfm_last_error": "WFMLastError",
    "wfm_completed_date": "WFMCompletedDate",
    "wfm_payload": "WFMPayload",
}

MAJOR_MOVE_METRES = 100.0
EDIT_BATCH_SIZE = 200
DRY_RUN = False
CREATE_WFM_JOBS = True
FULLY_AUTOMATE_WFM = True
WFM_JOB_TEMPLATE_ID = os.getenv("WFM_JOB_TEMPLATE_ID", "MFgMCmMtTQCNzs3dgSdeRA").strip()
WFM_ORG_ID = "SdBwj74JF24xKhpX"
WFM_BASE = f"https://workflow.arcgis.com/{WFM_ORG_ID}/{WORKFLOW_ITEM_ID}"
WFM_STEP_TIMEOUT_SECONDS = 5
MAX_INFERRED_DELETES_PER_RUN = 25
DELETE_DROP_FRACTION = 0.50
DELETE_DROP_MIN_ACTIVE = 5
MAX_WFM_RECONCILIATION_ATTEMPTS = 5
LEASE_MINUTES = 20
NOTEBOOK_ITEM_ID = (os.getenv("ARCGIS_NOTEBOOK_ITEM_ID") or os.getenv("NOTEBOOK_ITEM_ID") or os.getenv("NB_ITEM_ID") or "21aee7d6ee4746d8bfa236acea698148").strip()
LEASE_PROPERTY = "specialEventAuditLease"
HASH_EXCLUDE = {
    "objectid", "globalid", "creationdate", "creator", "editdate", "editor",
    "created_user", "created_date", "last_edited_user", "last_edited_date",
}

# ArcGIS Notebooks supplies the signed-in home portal. PORTAL_URL is documentation and
# an expected-org guard, not a second login.
gis = GIS("home")
source_item = gis.content.get(SOURCE_ITEM_ID)
audit_item = gis.content.get(AUDIT_ITEM_ID)
if source_item is None or SOURCE_LAYER_ID >= len(source_item.layers):
    raise RuntimeError(f"Source item/layer is unavailable: {SOURCE_ITEM_ID}/{SOURCE_LAYER_ID}")
if audit_item is None or AUDIT_LAYER_ID >= len(audit_item.layers):
    raise RuntimeError(f"Audit item/layer is unavailable: {AUDIT_ITEM_ID}/{AUDIT_LAYER_ID}")
source_layer = source_item.layers[SOURCE_LAYER_ID]
audit_layer = audit_item.layers[AUDIT_LAYER_ID]
if str(audit_layer.url).rstrip("/").lower() != AUDIT_LAYER_URL.rstrip("/").lower():
    raise RuntimeError(f"Audit item layer URL mismatch: {audit_layer.url}")

source_names = {f["name"].lower(): f["name"] for f in source_layer.properties.fields}
audit_names = {f["name"].lower(): f["name"] for f in audit_layer.properties.fields}
missing_source = [name for name in SOURCE_FIELDS.values() if name.lower() not in source_names]
missing_audit = [name for name in AUDIT_FIELDS.values() if name.lower() not in audit_names]
if missing_source or missing_audit:
    raise RuntimeError({"missing_source_fields": missing_source, "missing_audit_fields": missing_audit})
SOURCE_FIELDS = {k: source_names[v.lower()] for k, v in SOURCE_FIELDS.items()}
AUDIT_FIELDS = {k: audit_names[v.lower()] for k, v in AUDIT_FIELDS.items()}

SOURCE_OID_FIELD = source_layer.properties.objectIdField
SOURCE_GLOBALID_FIELD = getattr(source_layer.properties, "globalIdField", None)
edit_info = dict(getattr(source_layer.properties, "editFieldsInfo", {}) or {})
SOURCE_EDIT_DATE_FIELD = edit_info.get("editDateField")
SOURCE_EDITOR_FIELD = edit_info.get("editorField")
for system_field in (SOURCE_OID_FIELD, SOURCE_GLOBALID_FIELD, SOURCE_EDIT_DATE_FIELD, SOURCE_EDITOR_FIELD):
    if system_field:
        HASH_EXCLUDE.add(str(system_field).lower())
RUN_ID = str(uuid.uuid4())
LEASE_OWNER = f"{socket.gethostname()}:{os.getpid()}:{RUN_ID}"
print({"run_id": RUN_ID, "dry_run": DRY_RUN, "source": source_layer.url, "audit": audit_layer.url})

In [ ]:
def utc_now():
    return datetime.now(timezone.utc)


def epoch_ms(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return int(value)
    if isinstance(value, str):
        value = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    return int(value.timestamp() * 1000)


def chunks(values, size=EDIT_BATCH_SIZE):
    for start in range(0, len(values), size):
        yield values[start:start + size]


def query_all(layer, where="1=1", out_fields="*", return_geometry=True):
    ids = sorted((layer.query(where=where, return_ids_only=True) or {}).get("objectIds") or [])
    rows = []
    for batch in chunks(ids, 1000):
        result = layer.query(
            object_ids=",".join(map(str, batch)),
            out_fields=out_fields,
            return_geometry=return_geometry,
            return_all_records=False,
        )
        rows.extend(result.features)
    return rows


def require_success(response, operation):
    failures = []
    for bucket in ("addResults", "updateResults", "deleteResults"):
        for result in (response or {}).get(bucket, []) or []:
            if not result.get("success"):
                failures.append({"bucket": bucket, "result": result})
    if failures:
        raise RuntimeError(f"{operation} failed: {failures[:10]}")
    return response


def apply_adds(layer, features, label):
    if DRY_RUN:
        print(f"DRY_RUN {label}: adds={len(features)}")
        return
    for batch in chunks(features):
        require_success(layer.edit_features(adds=batch, rollback_on_failure=True), label)


def apply_updates(layer, features, label):
    if DRY_RUN:
        print(f"DRY_RUN {label}: updates={len(features)}")
        return
    for batch in chunks(features):
        require_success(layer.edit_features(updates=batch, rollback_on_failure=True), label)


def normalize_geometry(geometry):
    if not geometry:
        return None
    data = dict(geometry)
    return {
        "x": round(float(data["x"]), 8),
        "y": round(float(data["y"]), 8),
        "wkid": (data.get("spatialReference") or {}).get("latestWkid") or (data.get("spatialReference") or {}).get("wkid"),
    }


def canonical_state_hash(feature):
    attrs = {
        key.lower(): value for key, value in feature.attributes.items()
        if key.lower() not in HASH_EXCLUDE
    }
    payload = {"attributes": attrs, "geometry": normalize_geometry(feature.geometry)}
    text = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def parse_delivery_key(value):
    parts = str(value or "").split("|")
    if len(parts) == 4 and parts[0] == "v1":
        return {"change_type": parts[1], "state_hash": parts[2], "transition_hash": parts[3]}
    return None


def delivery_key(source_identity, previous_key, change_type, state_hash):
    transition = "|".join([str(source_identity), previous_key or "", change_type, state_hash])
    digest = hashlib.sha256(transition.encode("utf-8")).hexdigest()
    return f"v1|{change_type}|{state_hash}|{digest}"


def point_xy(geometry):
    data = normalize_geometry(geometry)
    return None if data is None else (data["x"], data["y"], data["wkid"])


def lonlat(geometry):
    point = point_xy(geometry)
    if not point:
        return None
    x, y, wkid = point
    if wkid in (4326, 4269):
        return x, y
    if wkid in (3857, 102100, 102113):
        return math.degrees(x / 6378137.0), math.degrees(2 * math.atan(math.exp(y / 6378137.0)) - math.pi / 2)
    return None


def distance_metres(old_geometry, new_geometry):
    a, b = lonlat(old_geometry), lonlat(new_geometry)
    if not a or not b:
        return None
    lon1, lat1, lon2, lat2 = map(math.radians, [a[0], a[1], b[0], b[1]])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371008.8 * 2 * math.asin(min(1.0, math.sqrt(h)))


def latest_by_source_objectid(audit_rows):
    latest = {}
    for feature in sorted(
        audit_rows,
        key=lambda f: (
            epoch_ms(f.attributes.get(AUDIT_FIELDS["change_time"])) or 0,
            int(f.attributes.get(AUDIT_FIELDS["oid"]) or 0),
        ),
    ):
        source_oid = feature.attributes.get(AUDIT_FIELDS["source_oid"])
        if source_oid is not None:
            latest[str(source_oid)] = feature
    return latest


def quoted(values):
    return ",".join("'" + str(value).replace("'", "''") + "'" for value in values)

In [ ]:
def _notebook_item():
    if not NOTEBOOK_ITEM_ID:
        return None
    try:
        return gis.content.get(NOTEBOOK_ITEM_ID)
    except Exception as exc:
        print(f"Lease disabled: notebook item could not be resolved: {exc}")
        return None


def _read_item_properties(item):
    raw = getattr(item, "properties", None)
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except Exception:
            return {}
    return dict(raw or {})


def _write_item_properties(item, properties):
    # Sharing API item properties are a JSON object represented as a JSON string by
    # some ArcGIS API for Python releases. Preserve unrelated properties.
    return item.update(item_properties={"properties": json.dumps(properties, separators=(",", ":"))})


class LeaseConflict(RuntimeError):
    pass


@contextmanager
def processing_lease():
    if DRY_RUN:
        print("DRY_RUN: notebook-item lease is not written.")
        yield
        return
    item = _notebook_item()
    if item is None:
        print("WARNING: no notebook item id/property access; proceeding without a lease. Prevent overlap in the scheduler.")
        yield
        return
    try:
        properties = _read_item_properties(item)
        lease = properties.get(LEASE_PROPERTY) or {}
        now = epoch_ms(utc_now())
        if lease.get("owner") and int(lease.get("expires", 0)) > now:
            raise LeaseConflict(f"Processor lease is held by {lease['owner']} until {lease['expires']}")
        properties[LEASE_PROPERTY] = {
            "owner": LEASE_OWNER,
            "expires": epoch_ms(utc_now() + timedelta(minutes=LEASE_MINUTES)),
            "runId": RUN_ID,
        }
        if not _write_item_properties(item, properties):
            raise RuntimeError("Notebook item property update returned false")
        verified = _read_item_properties(gis.content.get(NOTEBOOK_ITEM_ID)).get(LEASE_PROPERTY) or {}
        if verified.get("owner") != LEASE_OWNER:
            raise LeaseConflict("Notebook item lease verification lost a race")
    except LeaseConflict:
        raise
    except Exception as exc:
        print(f"WARNING: notebook-item lease is unavailable ({exc}); proceeding without it. Prevent overlap in the scheduler.")
        yield
        return
    try:
        yield
    finally:
        try:
            current_item = gis.content.get(NOTEBOOK_ITEM_ID)
            properties = _read_item_properties(current_item)
            if (properties.get(LEASE_PROPERTY) or {}).get("owner") == LEASE_OWNER:
                properties.pop(LEASE_PROPERTY, None)
                _write_item_properties(current_item, properties)
        except Exception as exc:
            print(f"WARNING: failed to clear notebook-item lease: {exc}")


def wfm_request(method, path, body=None):
    token = getattr(gis._con, "token", None)
    if not token:
        raise RuntimeError("The Notebook home connection did not expose a user token for Workflow Manager")
    response = requests.request(
        method,
        f"{WFM_BASE}/{path.lstrip('/')}",
        params={"token": token},
        json=body,
        timeout=60,
    )
    try:
        data = response.json()
    except Exception:
        raise RuntimeError(f"Workflow Manager returned HTTP {response.status_code}: {response.text[:500]}")
    if not response.ok or data.get("error"):
        raise RuntimeError(f"Workflow Manager {method} {path} failed: {json.dumps(data)[:1500]}")
    return data


def wfm_job(job_id):
    return wfm_request("GET", f"jobs/{job_id}?extProps=true")


def wfm_current_steps(job_id):
    return wfm_job(job_id).get("currentSteps") or []


def wfm_complete_step(job_id, step_id, answer=None, comment=None):
    # Manual, question, and start/end steps require the notification websocket's
    # StepInformation message; REST Finish returns WaitingForInfo for these steps.
    try:
        import websocket
    except ImportError as exc:
        raise RuntimeError("Install/enable websocket-client in the ArcGIS Notebook runtime") from exc
    username = getattr(gis.users.me, "username", None)
    current_info = next((row for row in wfm_current_steps(job_id) if row.get("stepId") == step_id), {})
    if current_info.get("assignedTo") != username or current_info.get("assignedType") != "User":
        try:
            wfm_request("POST", f"jobs/{job_id}/{step_id}", {"assignedType": "User", "assignedTo": username})
        except RuntimeError as exc:
            if "currently running" not in str(exc).lower():
                raise
    try:
        run_result = wfm_request("POST", f"jobs/{job_id}/action", {"type": "Run", "stepIds": [step_id]})
        rejected = [row for row in run_result.get("stepResponses", []) if row.get("status") not in (None, "Success")]
        if rejected and not all("running" in json.dumps(row).lower() for row in rejected):
            raise RuntimeError(f"Workflow Manager refused Run for {step_id}: {rejected}")
    except RuntimeError as exc:
        if "running" not in str(exc).lower():
            raise
    token = gis._con.token
    message = {"jobId": job_id, "stepId": step_id}
    if answer is None:
        message["manualResponse"] = True
    else:
        message["questionResponse"] = answer
    if comment:
        message["comment"] = comment[:2000]
    ws_url = f"wss://workflow.arcgis.com/{WFM_ORG_ID}/{WORKFLOW_ITEM_ID}/notificationWs?token={quote(token, safe='')}"
    ws = websocket.create_connection(ws_url, timeout=2)
    try:
        ws.send(json.dumps({"msgType": "subscribe", "jobIds": [job_id], "token": token}))
        ws.send(json.dumps({"msgType": "StepInformation", "message": message, "token": token}))
        deadline = time.time() + WFM_STEP_TIMEOUT_SECONDS
        while time.time() < deadline:
            try:
                event = json.loads(ws.recv())
            except websocket.WebSocketTimeoutException:
                current = wfm_current_steps(job_id)
                if not any(step.get("stepId") == step_id for step in current):
                    return current
                continue
            if event.get("msgType") == "StepFinished" and step_id in (event.get("message", {}).get("stepIds") or []):
                return event.get("message", {}).get("currentSteps") or []
            if event.get("msgType") in ("StepError", "StepInfoRequiredError"):
                raise RuntimeError(f"Workflow Manager step error: {event}")
    finally:
        ws.close()
    # A completion event can be delayed or lost. Re-read before treating the timeout
    # as a failure so the Notebook never submits the same response twice.
    current = wfm_current_steps(job_id)
    if any(step.get("stepId") == step_id for step in current):
        raise RuntimeError(f"Timed out completing Workflow Manager step {step_id}")
    return current


def wfm_add_comment(job_id, text):
    return wfm_request("POST", f"jobs/{job_id}/comments", {"comment": text[:4000]})


def wfm_location(geometry):
    data = normalize_geometry(geometry)
    if not data:
        return None
    shape = {"points": [[data["x"], data["y"]]], "spatialReference": {"wkid": data["wkid"]}}
    return {"geometryType": "Multipoint", "geometry": json.dumps(shape, separators=(",", ":"))}


def wfm_run_to_close(job_id, exception, payload_text):
    exception_type = "DELETE" if exception["change_type"] == "DELETE" else "MAJOR_MOVE"
    responses = {
        "exception-created": None,
        "review": None,
        "valid": 1,
        "escalate": None,
        # A delete already satisfies the requested deletion. A major-move audit is
        # recorded as requiring no additional source-data change.
        "required-action": 2 if exception_type == "DELETE" else 3,
        "wait-delete": None,
        "verify-delete": None,
        "wait-correction": None,
        "verify-correction": None,
        "resolve": None,
        "dismiss": None,
        "closed": None,
    }
    for _ in range(16):
        current = wfm_current_steps(job_id)
        if not current:
            break
        step_id = current[0].get("stepId")
        if step_id not in responses:
            raise RuntimeError(f"Unexpected Workflow Manager step {step_id} for job {job_id}")
        step_comment = payload_text if step_id in ("review", "resolve") else None
        wfm_complete_step(job_id, step_id, answer=responses[step_id], comment=step_comment)
        if step_id == "closed":
            break
    wfm_request("POST", "jobs/manage", {"type": "Close", "jobIds": [job_id]})
    return wfm_job(job_id)


def wfm_find_job(job_name):
    safe_name = str(job_name).replace("'", "''")
    fields = ["jobId", "jobName", "closed", "currentStep", "currentStepId"]
    result = wfm_request("POST", "jobs/search", {"q": f"jobName = '{safe_name}'", "fields": fields, "num": 10})
    names = [field.get("name") if isinstance(field, dict) else field for field in result.get("fields", fields)]
    rows = [dict(zip(names, row)) for row in result.get("results", [])]
    return rows[0] if rows else None


def audit_delivery_row(delivery_key_value):
    where = f"{AUDIT_FIELDS['delivery_key']} = {quoted([delivery_key_value])}"
    rows = query_all(audit_layer, where=where, out_fields="*", return_geometry=True)
    return rows[0] if rows else None


def update_audit_delivery(delivery_key_value, **values):
    row = audit_delivery_row(delivery_key_value)
    if row is None:
        raise RuntimeError(f"Audit row not found for DeliveryKey {delivery_key_value}")
    attrs = {AUDIT_FIELDS["oid"]: row.attributes[AUDIT_FIELDS["oid"]]}
    attrs.update(values)
    apply_updates(audit_layer, [{"attributes": attrs}], "update WFM delivery state")
    return attrs


def create_wfm_job(exception):
    if not CREATE_WFM_JOBS:
        return None
    if not WFM_JOB_TEMPLATE_ID:
        raise RuntimeError("Set WFM_JOB_TEMPLATE_ID before CREATE_WFM_JOBS=True")
    source_oid = exception["source_objectid"]
    change_type = exception["change_type"]
    exception_type = "DELETE" if change_type == "DELETE" else "MAJOR_MOVE"
    distance = exception.get("distance_metres")
    delivery_key_value = str(exception["delivery_key"])
    audit_id = hashlib.sha256(delivery_key_value.encode("utf-8")).hexdigest()
    payload = {
        "audit_id": audit_id,
        "asset_id": str(source_oid),
        "event_id": str(exception.get("event_id") or ""),
        "exception_type": exception_type,
        "change_time_utc": exception.get("change_time_utc") or utc_now().isoformat(),
        "distance_m": None if distance is None else round(float(distance), 1),
        "delivery_key": delivery_key_value,
        "source_layer": source_layer.url,
        "notebook_run_id": RUN_ID,
    }
    payload_text = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    prior = audit_delivery_row(delivery_key_value)
    prior_attempts = int((prior.attributes.get(AUDIT_FIELDS["wfm_attempt_count"]) if prior else 0) or 0)
    attempt = prior_attempts + 1
    update_audit_delivery(
        delivery_key_value,
        **{
            AUDIT_FIELDS["wfm_status"]: "PENDING",
            AUDIT_FIELDS["wfm_attempt_count"]: attempt,
            AUDIT_FIELDS["wfm_last_error"]: None,
            AUDIT_FIELDS["wfm_payload"]: payload_text[:4000],
        },
    )
    job_name = f"SEA_AUTO_{audit_id[:16]}"
    job_id = None
    try:
        existing = wfm_find_job(job_name)
        if existing:
            job_id = existing.get("jobId")
            if existing.get("closed") in (True, 1, "1"):
                update_audit_delivery(delivery_key_value, **{
                    AUDIT_FIELDS["wfm_job_id"]: job_id,
                    AUDIT_FIELDS["wfm_status"]: "CLOSED",
                    AUDIT_FIELDS["wfm_completed_date"]: epoch_ms(utc_now()),
                    AUDIT_FIELDS["wfm_last_error"]: None,
                })
                return {"jobId": job_id, "closed": True, "payload": payload, "resumed": True}
        else:
            description = f"{exception_type} exception for source ObjectID {source_oid}. Notebook payload={payload_text}"
            extended_properties = [
                {"identifier": "audit_exception.audit_id", "value": audit_id},
                {"identifier": "audit_exception.asset_id", "value": str(source_oid)},
                {"identifier": "audit_exception.event_id", "value": payload["event_id"]},
                {"identifier": "audit_exception.exception_type", "value": exception_type},
                {"identifier": "audit_exception.change_time_utc", "value": payload["change_time_utc"]},
                {"identifier": "audit_exception.distance_m", "value": "" if distance is None else f"{float(distance):.1f}"},
                {"identifier": "audit_exception.details", "value": payload_text[:2000]},
                {"identifier": "audit_exception.resolution", "value": "Automatically processed and closed from the scheduled Notebook payload."},
            ]
            body = {
                "numberOfJobs": 1,
                "jobName": job_name,
                "description": description[:2000],
                "extendedProperties": extended_properties,
            }
            location = wfm_location(exception.get("geometry"))
            if location:
                body["location"] = location
            created = wfm_request("POST", f"jobTemplates/{WFM_JOB_TEMPLATE_ID}/job", body)
            job_ids = created.get("jobIds") or []
            if not job_ids:
                raise RuntimeError(f"Workflow Manager returned no job id: {created}")
            job_id = job_ids[0]
            wfm_add_comment(job_id, f"Notebook audit payload: {payload_text}")
        update_audit_delivery(delivery_key_value, **{
            AUDIT_FIELDS["wfm_job_id"]: job_id,
            AUDIT_FIELDS["wfm_status"]: "PROCESSING",
        })
        final_job = wfm_run_to_close(job_id, exception, payload_text) if FULLY_AUTOMATE_WFM else wfm_job(job_id)
        closed = bool(final_job.get("closed"))
        update_audit_delivery(delivery_key_value, **{
            AUDIT_FIELDS["wfm_job_id"]: job_id,
            AUDIT_FIELDS["wfm_status"]: "CLOSED" if closed else "PROCESSING",
            AUDIT_FIELDS["wfm_completed_date"]: epoch_ms(utc_now()) if closed else None,
            AUDIT_FIELDS["wfm_last_error"]: None,
        })
        return {"jobId": job_id, "closed": closed, "payload": payload, "resumed": bool(existing)}
    except Exception as exc:
        update_audit_delivery(delivery_key_value, **{
            AUDIT_FIELDS["wfm_job_id"]: job_id,
            AUDIT_FIELDS["wfm_status"]: "FAILED",
            AUDIT_FIELDS["wfm_last_error"]: str(exc)[:2000],
        })
        raise


def reconcile_wfm_deliveries():
    if not CREATE_WFM_JOBS or DRY_RUN:
        return [], []
    status_field = AUDIT_FIELDS["wfm_status"]
    payload_field = AUDIT_FIELDS["wfm_payload"]
    attempts_field = AUDIT_FIELDS["wfm_attempt_count"]
    where = (
        f"{status_field} IN ('PENDING','PROCESSING','FAILED') AND {payload_field} IS NOT NULL "
        f"AND ({attempts_field} IS NULL OR {attempts_field} < {MAX_WFM_RECONCILIATION_ATTEMPTS})"
    )
    pending = query_all(audit_layer, where=where, out_fields="*", return_geometry=True)
    results, failures = [], []
    for row in pending:
        try:
            payload = json.loads(row.attributes.get(payload_field) or "{}")
            exception_type = str(payload.get("exception_type") or "").upper()
            exception = {
                "source_objectid": payload.get("asset_id") or row.attributes.get(AUDIT_FIELDS["source_oid"]),
                "event_id": payload.get("event_id") or row.attributes.get(AUDIT_FIELDS["event_id"]),
                "change_type": "DELETE" if exception_type == "DELETE" else "UPDATE",
                "distance_metres": payload.get("distance_m"),
                "delivery_key": payload.get("delivery_key") or row.attributes.get(AUDIT_FIELDS["delivery_key"]),
                "change_time_utc": payload.get("change_time_utc"),
                "geometry": dict(row.geometry) if row.geometry else None,
            }
            results.append(create_wfm_job(exception))
        except Exception as exc:
            failures.append({"deliveryKey": row.attributes.get(AUDIT_FIELDS["delivery_key"]), "error": str(exc)})
    return results, failures

In [ ]:
def make_audit_feature(current, previous, change_type, state_hash, key, now_ms):
    current_attrs = current.attributes if current else {}
    previous_attrs = previous.attributes if previous else {}
    attrs = current_attrs if current else previous_attrs
    source_oid = current_attrs.get(SOURCE_OID_FIELD) if current else previous_attrs.get(AUDIT_FIELDS["source_oid"])
    source_gid = (
        current_attrs.get(SOURCE_GLOBALID_FIELD) if current and SOURCE_GLOBALID_FIELD
        else previous_attrs.get(AUDIT_FIELDS["source_global_id"])
    )
    change_time = current_attrs.get(SOURCE_EDIT_DATE_FIELD) if current and SOURCE_EDIT_DATE_FIELD else now_ms
    editor = current_attrs.get(SOURCE_EDITOR_FIELD) if current and SOURCE_EDITOR_FIELD else getattr(gis.users.me, "username", None)
    values = {
        AUDIT_FIELDS["source_oid"]: source_oid,
        AUDIT_FIELDS["source_global_id"]: source_gid,
        AUDIT_FIELDS["event_id"]: current_attrs.get(SOURCE_FIELDS["event_id"]) if current else previous_attrs.get(AUDIT_FIELDS["event_id"]),
        AUDIT_FIELDS["unit_id"]: current_attrs.get(SOURCE_FIELDS["unit_id"]) if current else previous_attrs.get(AUDIT_FIELDS["unit_id"]),
        AUDIT_FIELDS["event_asset_id"]: current_attrs.get(SOURCE_FIELDS["event_asset_id"]) if current else previous_attrs.get(AUDIT_FIELDS["event_asset_id"]),
        AUDIT_FIELDS["unit_location"]: current_attrs.get(SOURCE_FIELDS["unit_location"]) if current else previous_attrs.get(AUDIT_FIELDS["unit_location"]),
        AUDIT_FIELDS["unit_type"]: current_attrs.get(SOURCE_FIELDS["unit_type"]) if current else previous_attrs.get(AUDIT_FIELDS["unit_type"]),
        AUDIT_FIELDS["change_type"]: change_type,
        AUDIT_FIELDS["change_time"]: epoch_ms(change_time),
        AUDIT_FIELDS["editor_user"]: editor,
        AUDIT_FIELDS["processed_time"]: now_ms,
        AUDIT_FIELDS["delivery_key"]: key,
        AUDIT_FIELDS["source_url"]: source_layer.url,
        AUDIT_FIELDS["is_last_known"]: 1,
        AUDIT_FIELDS["wfm_job_id"]: None,
        AUDIT_FIELDS["wfm_status"]: None,
        AUDIT_FIELDS["wfm_attempt_count"]: 0,
        AUDIT_FIELDS["wfm_last_error"]: None,
        AUDIT_FIELDS["wfm_completed_date"]: None,
        AUDIT_FIELDS["wfm_payload"]: None,
    }
    geometry = current.geometry if current else previous.geometry
    return {"attributes": values, "geometry": dict(geometry) if geometry else None}


def run_processor():
    now_ms = epoch_ms(utc_now())
    source_rows = query_all(source_layer, out_fields="*", return_geometry=True)
    audit_rows = query_all(audit_layer, out_fields="*", return_geometry=True)
    latest = latest_by_source_objectid(audit_rows)

    current = {}
    for feature in source_rows:
        source_oid = feature.attributes.get(SOURCE_OID_FIELD)
        if source_oid is None:
            raise RuntimeError("A source row has no ObjectID")
        key = str(source_oid)
        if key in current:
            raise RuntimeError(f"Duplicate source ObjectID {source_oid}")
        current[key] = feature

    reconciled_results, reconciliation_failures = reconcile_wfm_deliveries()
    adds, superseded, exceptions = [], [], []
    candidate_meta = []
    all_keys = {str(f.attributes.get(AUDIT_FIELDS["delivery_key"])) for f in audit_rows if f.attributes.get(AUDIT_FIELDS["delivery_key"])}

    for source_oid, feature in current.items():
        previous = latest.get(source_oid)
        previous_type = str(previous.attributes.get(AUDIT_FIELDS["change_type"]) or "").upper() if previous else None
        previous_key = previous.attributes.get(AUDIT_FIELDS["delivery_key"]) if previous else None
        previous_state = parse_delivery_key(previous_key)
        state_hash = canonical_state_hash(feature)
        if previous and previous_type != "DELETE" and previous_state and previous_state["state_hash"] == state_hash:
            continue
        change_type = "ADD" if previous is None or previous_type == "DELETE" else "UPDATE"
        identity = feature.attributes.get(SOURCE_GLOBALID_FIELD) if SOURCE_GLOBALID_FIELD else None
        identity = identity or source_oid
        key = delivery_key(identity, previous_key, change_type, state_hash)
        if key in all_keys:
            continue
        distance = distance_metres(previous.geometry, feature.geometry) if previous and previous_type != "DELETE" else None
        audit_feature = make_audit_feature(feature, previous, change_type, state_hash, key, now_ms)
        adds.append(audit_feature)
        candidate_meta.append((audit_feature, previous, distance))
        if previous:
            superseded.append({"attributes": {
                AUDIT_FIELDS["oid"]: previous.attributes[AUDIT_FIELDS["oid"]],
                AUDIT_FIELDS["is_last_known"]: 0,
            }})
        if distance is not None and distance >= MAJOR_MOVE_METRES:
            exceptions.append({
                "source_objectid": source_oid,
                "event_id": feature.attributes.get(SOURCE_FIELDS["event_id"]),
                "change_type": change_type,
                "distance_metres": distance,
                "delivery_key": key,
                "geometry": dict(feature.geometry) if feature.geometry else None,
            })
        all_keys.add(key)

    previous_active_keys = {
        source_oid for source_oid, row in latest.items()
        if str(row.attributes.get(AUDIT_FIELDS["change_type"]) or "").upper() != "DELETE"
    }
    inferred_delete_keys = previous_active_keys.difference(current.keys())
    inferred_delete_count = len(inferred_delete_keys)
    inferred_delete_fraction = (inferred_delete_count / len(previous_active_keys)) if previous_active_keys else 0.0
    excessive_count = inferred_delete_count > MAX_INFERRED_DELETES_PER_RUN
    excessive_fraction = (
        len(previous_active_keys) >= DELETE_DROP_MIN_ACTIVE
        and inferred_delete_count >= DELETE_DROP_MIN_ACTIVE
        and inferred_delete_fraction >= DELETE_DROP_FRACTION
    )
    if excessive_count or excessive_fraction:
        raise RuntimeError(
            f"Delete circuit breaker: refusing {inferred_delete_count} inferred deletes from "
            f"{len(previous_active_keys)} previously active assets ({inferred_delete_fraction:.1%})"
        )

    for source_oid, previous in latest.items():
        if source_oid in current:
            continue
        previous_type = str(previous.attributes.get(AUDIT_FIELDS["change_type"]) or "").upper()
        if previous_type == "DELETE":
            continue
        previous_key = previous.attributes.get(AUDIT_FIELDS["delivery_key"])
        previous_state = parse_delivery_key(previous_key)
        state_hash = previous_state["state_hash"] if previous_state else hashlib.sha256(str(previous_key).encode("utf-8")).hexdigest()
        identity = previous.attributes.get(AUDIT_FIELDS["source_global_id"]) or source_oid
        key = delivery_key(identity, previous_key, "DELETE", state_hash)
        if key in all_keys:
            continue
        audit_feature = make_audit_feature(None, previous, "DELETE", state_hash, key, now_ms)
        adds.append(audit_feature)
        superseded.append({"attributes": {
            AUDIT_FIELDS["oid"]: previous.attributes[AUDIT_FIELDS["oid"]],
            AUDIT_FIELDS["is_last_known"]: 0,
        }})
        exceptions.append({
            "source_objectid": source_oid,
            "event_id": previous.attributes.get(AUDIT_FIELDS["event_id"]),
            "change_type": "DELETE",
            "distance_metres": None,
            "delivery_key": key,
            "geometry": dict(previous.geometry) if previous.geometry else None,
        })
        all_keys.add(key)

    # Query candidate DeliveryKeys immediately before append so a retry or competing run
    # cannot intentionally add a key already visible in the service. A unique index remains
    # the only complete protection against a simultaneous write race.
    if adds:
        keys = [row["attributes"][AUDIT_FIELDS["delivery_key"]] for row in adds]
        existing = set()
        for batch in chunks(keys, 10):
            where = f"{AUDIT_FIELDS['delivery_key']} IN ({quoted(batch)})"
            existing.update(
                str(f.attributes[AUDIT_FIELDS["delivery_key"]])
                for f in query_all(audit_layer, where=where, out_fields=AUDIT_FIELDS["delivery_key"], return_geometry=False)
            )
        keep = {key for key in keys if key not in existing}
        adds = [row for row in adds if row["attributes"][AUDIT_FIELDS["delivery_key"]] in keep]
        exceptions = [row for row in exceptions if row["delivery_key"] in keep]
        keep_previous_oids = {
            previous.attributes[AUDIT_FIELDS["oid"]]
            for row, previous, _distance in candidate_meta
            if previous and row["attributes"][AUDIT_FIELDS["delivery_key"]] in keep
        }
        delete_previous_oids = {
            latest[str(row["source_objectid"])].attributes[AUDIT_FIELDS["oid"]]
            for row in exceptions if row["change_type"] == "DELETE" and str(row["source_objectid"]) in latest
        }
        keep_previous_oids.update(delete_previous_oids)
        superseded = [row for row in superseded if row["attributes"][AUDIT_FIELDS["oid"]] in keep_previous_oids]

    # Persist every qualifying Workflow Manager payload before the audit append. If this
    # run times out while processing several jobs, the next run can reconcile every row,
    # including exceptions it never reached in the per-job loop.
    add_by_key = {row["attributes"][AUDIT_FIELDS["delivery_key"]]: row for row in adds}
    for exception in exceptions:
        key = str(exception["delivery_key"])
        audit_id = hashlib.sha256(key.encode("utf-8")).hexdigest()
        distance = exception.get("distance_metres")
        payload = {
            "audit_id": audit_id,
            "asset_id": str(exception["source_objectid"]),
            "event_id": str(exception.get("event_id") or ""),
            "exception_type": "DELETE" if exception["change_type"] == "DELETE" else "MAJOR_MOVE",
            "change_time_utc": utc_now().isoformat(),
            "distance_m": None if distance is None else round(float(distance), 1),
            "delivery_key": key,
            "source_layer": source_layer.url,
            "notebook_run_id": RUN_ID,
        }
        if key in add_by_key:
            add_by_key[key]["attributes"][AUDIT_FIELDS["wfm_status"]] = "PENDING"
            add_by_key[key]["attributes"][AUDIT_FIELDS["wfm_payload"]] = json.dumps(payload, sort_keys=True, separators=(",", ":"))[:4000]

    # Append events first. Clearing IsLastKnown on prior rows is the only asset-state mutation;
    # WFM fields are durable delivery state and latest asset reconstruction ignores them.
    apply_adds(audit_layer, adds, "audit append")
    apply_updates(audit_layer, superseded, "clear prior IsLastKnown")
    workflow_results = list(reconciled_results)
    workflow_failures = list(reconciliation_failures)
    if not DRY_RUN:
        for exception in exceptions:
            try:
                workflow_results.append(create_wfm_job(exception))
            except Exception as exc:
                workflow_failures.append({"deliveryKey": exception.get("delivery_key"), "error": str(exc)})

    summary = {
        "source_count": len(source_rows),
        "audit_count_before": len(audit_rows),
        "audit_adds": len(adds),
        "prior_flags_cleared": len(superseded),
        "inferred_delete_count": inferred_delete_count,
        "inferred_delete_fraction": round(inferred_delete_fraction, 4),
        "workflow_exceptions": len(exceptions),
        "workflow_reconciled": len([row for row in reconciled_results if row]),
        "workflow_jobs_processed": len([row for row in workflow_results if row]),
        "workflow_jobs_closed": len([row for row in workflow_results if row and row.get("closed")]),
        "workflow_job_ids": [row.get("jobId") for row in workflow_results if row],
        "workflow_failures": workflow_failures,
        "dry_run": DRY_RUN,
    }
    print(json.dumps(summary, indent=2))
    if workflow_failures:
        raise RuntimeError(f"Workflow Manager delivery failures: {workflow_failures[:10]}")
    return summary

In [ ]:
def publish_run_status(status, summary=None, error=None):
    item = _notebook_item()
    if item is None:
        return
    props = dict(item.properties or {})
    props["specialEventAuditLastRun"] = {
        "status": status,
        "runId": RUN_ID,
        "timeUtc": utc_now().isoformat(),
        "summary": summary,
        "error": None if error is None else str(error)[:2000],
    }
    item.update(item_properties={"properties": props})


# Scheduled-task entry point. Persist a compact health record on the Notebook item
# so operations can monitor the last completed run without opening the output notebook.
try:
    with processing_lease():
        RESULT = run_processor()
    publish_run_status("SUCCEEDED", summary=RESULT)
except Exception as exc:
    publish_run_status("FAILED", error=exc)
    raise
RESULT

## Operational notes and limitations

- **One audit layer only:** there are no MovementSegments, AssetSnapshot, or IntegrationState datasets. Current state is reconstructed by sorting `ChangeDateTime`, then `OBJECTID`, and taking the last row for each `SourceObjectID`.
- **Source changes:** the state hash includes every non-system source attribute plus normalized point geometry. This detects changes to `unitid`, `eventid`, `uniqueid`, `unitloc`, `unittype`, `poststatus`, `staffagency`, and any additional business attributes even though the audit layer stores only a subset directly.
- **Idempotency:** `DeliveryKey` embeds the canonical state hash and a deterministic transition hash. The processor queries candidate keys before append, and the audit layer has a unique `UX_Audit_DeliveryKey` index to close the simultaneous-write race.
- **Append-only semantics:** audit events and geometry are appended and never rewritten. The previous row's `IsLastKnown` flag and the current row's WFM delivery fields are administrative state updates. Time/OBJECTID ordering still reconstructs the latest asset state.
- **Deletes:** a delete is inferred only after a complete paged source read and stores the prior audit point as last-known geometry. The processor refuses more than 25 inferred deletes or a drop of at least 50 percent when at least five assets were active.
- **Movement:** distance is geodesic for WKID 4326/4269 and Web Mercator. Other spatial references return no distance and therefore do not trigger the 100 m rule until a validated projection step is added.
- **Lease:** the processor prefers a lease in the Notebook item's `properties` when an item id is available through `ARCGIS_NOTEBOOK_ITEM_ID`, `NOTEBOOK_ITEM_ID`, or `NB_ITEM_ID`. If property access is unavailable, it logs a warning and relies on scheduler overlap prevention.
- **Workflow Manager:** qualifying DELETE and major-movement events persist the WFM payload, job ID, status, attempt count, error, and completion date on the audit row. Deterministic job names prevent duplicate job creation; failed, pending, and processing rows are retried for up to five scheduled runs, resuming the existing job when present.
- **Live validation:** source and audit schemas, Web Mercator spatial references, edit capabilities, the unique DeliveryKey index, a successful scheduled run, and a complete test WFM close path were verified in the target organization. Synthetic source-data lifecycle testing is still required before operational use.